# Gmm Exploration
Original workflow: `GPN_V3_GMM.ipynb`. Imports now use the `amgpn` package.
Run from the repository root after an editable install. Configuration keys retain the original notebook/cell identifiers.


In [ ]:
# GPN_V3 使用GMM混合高斯原型模型的示例代码
'\n这个文件展示如何在GPN_V3.ipynb中调整参数以使用GMM混合高斯原型模型\n\n前提条件：\n<configured>. 将 loss_ada_num.py 替换为 loss_ada_num_gmm.py\n<configured>. 按照 GPN_V4_GMM_patch.py 中的说明修改 GPN_V4_adaMulti.py\n'

# ============================================
# 示例<configured>: 基础GMM模式
# ============================================
"""
在GPN_V3.ipynb的main函数或inference函数中，
将GPNTrainer的初始化代码修改为：
"""
from amgpn.config import require as _cfg_require, resolve as _cfg_resolve, format_value as _cfg_format

from amgpn.legacy.v4.adaptive_exploratory import GPN_Optimized, GPNTrainer

# 创建模型
model = GPN_Optimized(
    attention_type=_cfg_require('GPN_V3_GMM.ipynb.cell0.module.attention_type'),  # se, decoupled, freq_priority
    reduction=_cfg_require('GPN_V3_GMM.ipynb.cell0.module.reduction'),
)

# 创建trainer - GMM模式
meta_learner = GPNTrainer(
    model,
    device=device,
    mlr=_cfg_require('GPN_V3_GMM.ipynb.cell0.module.mlr'),                    # 学习率
    use_gmm=_cfg_require('GPN_V3_GMM.ipynb.cell0.module.use_gmm'),                 # ★ 启用GMM混合高斯模式
    gmm_components=_cfg_require('GPN_V3_GMM.ipynb.cell0.module.gmm_components'),             # ★ 每个类使用<configured>个高斯分量
    prototypes_per_class=_cfg_require('GPN_V3_GMM.ipynb.cell0.module.prototypes_per_class'),      # 最大原型数（向后兼容）
    cut_ratio=_cfg_require('GPN_V3_GMM.ipynb.cell0.module.cut_ratio'),                # 频率裁剪比例
    resize=_cfg_require('GPN_V3_GMM.ipynb.cell0.module.resize'),                  # 是否resize
    save_middle=_cfg_require('GPN_V3_GMM.ipynb.cell0.module.save_middle'),             # 是否保存中间最优模型
)

print('GMM模式已启用：每个类使用<configured>个高斯分量建模')

# 训练或评估
# meta_learner.train_loop(...)
# results = meta_learner.full_evaluation(...)

In [ ]:
# ============================================
# 示例<configured>: 调整GMM分量数的实验
# ============================================
"""
不同分量数的对比实验：
"""
from amgpn.config import require as _cfg_require, resolve as _cfg_resolve, format_value as _cfg_format

# 实验配置
gmm_configs = [
    {'components': 3, 'name': 'GMM-<configured>'},
    {'components': 5, 'name': 'GMM-<configured>'},
    {'components': 8, 'name': 'GMM-<configured>'},
    {'components': 10, 'name': 'GMM-<configured>'},
]

results_dict = {}

for config in gmm_configs:
    print(f"\n{'='*50}")
    print(f"实验: {config['name']}")
    print(f"{'='*50}")
    
    # 创建模型
    model = GPN_Optimized(attention_type=_cfg_require('GPN_V3_GMM.ipynb.cell1.module.attention_type'), reduction=_cfg_require('GPN_V3_GMM.ipynb.cell1.module.reduction'))
    
    # 创建trainer
    meta_learner = GPNTrainer(
        model,
        device=device,
        mlr=_cfg_require('GPN_V3_GMM.ipynb.cell1.module.mlr'),
        use_gmm=_cfg_require('GPN_V3_GMM.ipynb.cell1.module.use_gmm'),
        gmm_components=config['components'],  # ★ 调整分量数
        cut_ratio=_cfg_require('GPN_V3_GMM.ipynb.cell1.module.cut_ratio'),
        resize=_cfg_require('GPN_V3_GMM.ipynb.cell1.module.resize'),
    )
    
    # 评估
    results = meta_learner.full_evaluation(
        test_dataset, 
        n_trials=_cfg_require('GPN_V3_GMM.ipynb.cell1.module.n_trials'),
        q_query=_cfg_require('GPN_V3_GMM.ipynb.cell1.module.q_query')
    )
    
    results_dict[config['name']] = results
    
    print(f"{config['name']} 准确率: {results['mean_accuracy']:.4f} ± {results['std_accuracy']:.4f}")

# 绘制对比图
import matplotlib.pyplot as plt

names = [config['name'] for config in gmm_configs]
accuracies = [results_dict[name]['mean_accuracy'] for name in names]
stds = [results_dict[name]['std_accuracy'] for name in names]

plt.figure(figsize=(10, 6))
plt.errorbar(range(len(names)), accuracies, yerr=stds, marker='o', capsize=5)
plt.xticks(range(len(names)), names)
plt.ylabel('Accuracy')
plt.xlabel('GMM Configuration')
plt.title('GMM Components Number Comparison')
plt.grid(True, alpha=_cfg_require('GPN_V3_GMM.ipynb.cell1.module.alpha'))
plt.savefig('gmm_components_comparison.png', dpi=_cfg_require('GPN_V3_GMM.ipynb.cell1.module.size_or_budget'), bbox_inches='tight')
plt.show()

In [ ]:
# ============================================
# 示例<configured>: GMM vs 单原型 vs 多原型对比
# ============================================
"""
对比三种模式的性能：
"""
from amgpn.config import require as _cfg_require, resolve as _cfg_resolve, format_value as _cfg_format

# 配置列表
mode_configs = [
    {
        'name': '单原型模式',
        'use_gmm': _cfg_require('GPN_V3_GMM.ipynb.cell2.module.use_gmm'),
        'use_multi': _cfg_require('GPN_V3_GMM.ipynb.cell2.module.use_multi'),
        'gmm_components': _cfg_require('GPN_V3_GMM.ipynb.cell2.module.gmm_components'),
    },
    {
        'name': '多原型模式',
        'use_gmm': _cfg_require('GPN_V3_GMM.ipynb.cell2.module.use_gmm__2'),
        'use_multi': _cfg_require('GPN_V3_GMM.ipynb.cell2.module.use_multi__2'),
        'gmm_components': _cfg_require('GPN_V3_GMM.ipynb.cell2.module.gmm_components__2'),
        'merge_threshold': _cfg_require('GPN_V3_GMM.ipynb.cell2.module.merge_threshold'),
        'merge_start_epoch': _cfg_require('GPN_V3_GMM.ipynb.cell2.module.merge_start_epoch'),
    },
    {
        'name': 'GMM-<configured>分量',
        'use_gmm': _cfg_require('GPN_V3_GMM.ipynb.cell2.module.use_gmm__3'),
        'use_multi': _cfg_require('GPN_V3_GMM.ipynb.cell2.module.use_multi__3'),
        'gmm_components': _cfg_require('GPN_V3_GMM.ipynb.cell2.module.gmm_components__3'),
    },
    {
        'name': 'GMM-<configured>分量',
        'use_gmm': _cfg_require('GPN_V3_GMM.ipynb.cell2.module.use_gmm__4'),
        'use_multi': _cfg_require('GPN_V3_GMM.ipynb.cell2.module.use_multi__4'),
        'gmm_components': _cfg_require('GPN_V3_GMM.ipynb.cell2.module.gmm_components__4'),
    },
]

comparison_results = {}

for config in mode_configs:
    print(f"\n{'='*50}")
    print(f"模式: {config['name']}")
    print(f"{'='*50}")
    
    # 创建模型
    model = GPN_Optimized(attention_type=_cfg_require('GPN_V3_GMM.ipynb.cell2.module.attention_type'), reduction=_cfg_require('GPN_V3_GMM.ipynb.cell2.module.reduction'))
    
    # 创建trainer
    trainer_params = {
        'model': model,
        'device': device,
        'mlr': _cfg_require('GPN_V3_GMM.ipynb.cell2.module.mlr'),
        'use_gmm': config['use_gmm'],
        'use_multi': config['use_multi'],
        'cut_ratio': _cfg_require('GPN_V3_GMM.ipynb.cell2.module.cut_ratio'),
        'resize': _cfg_require('GPN_V3_GMM.ipynb.cell2.module.resize'),
        'save_middle': _cfg_require('GPN_V3_GMM.ipynb.cell2.module.save_middle'),
    }
    
    if config['use_gmm']:
        trainer_params['gmm_components'] = config['gmm_components']
    
    if config['use_multi']:
        trainer_params['merge_threshold'] = config.get('merge_threshold', 0.5)
        trainer_params['merge_start_epoch'] = config.get('merge_start_epoch', 0)
    
    meta_learner = GPNTrainer(**trainer_params)
    
    # 加载预训练模型（如果有）
    # Historical configuration omitted; provide values through private configuration.
    
    # 评估
    results = meta_learner.full_evaluation(
        test_dataset,
        n_trials=_cfg_require('GPN_V3_GMM.ipynb.cell2.module.n_trials'),  # 增加试验次数以获得更稳定的结果
        q_query=_cfg_require('GPN_V3_GMM.ipynb.cell2.module.q_query')
    )
    
    comparison_results[config['name']] = results
    
    print(f"准确率: {results['mean_accuracy']:.4f} ± {results['std_accuracy']:.4f}")
    print(f"推理时间: {results.get('avg_inference_time', 0):.4f}s")

# 生成对比表格
print(f"\n{'='*70}")
print("性能对比汇总")
print(f"{'='*70}")
print(f"{'模式':<15} {'准确率':<20} {'标准差':<15}")
print(f"{'-'*70}")
for name, result in comparison_results.items():
    acc = result['mean_accuracy']
    std = result['std_accuracy']
    print(f"{name:<15} {acc:.4f} ({acc*100:.2f}%)    ±{std:.4f}")

In [ ]:
# ============================================
# 示例<configured>: 完整的训练+评估流程（GMM模式）
# ============================================
"""
使用GMM模式进行完整的训练和评估：
"""
from amgpn.config import require as _cfg_require, resolve as _cfg_resolve, format_value as _cfg_format

def main():
    # 设置随机种子
    import random
    import numpy as np
    random.seed(_cfg_require('GPN_V3_GMM.ipynb.cell3.main.seed'))
    np.random.seed(_cfg_require('GPN_V3_GMM.ipynb.cell3.main.seed__2'))
    torch.manual_seed(_cfg_require('GPN_V3_GMM.ipynb.cell3.main.seed__3'))
    
    # 设备配置
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"使用设备: {device}")
    
    # 数据集参数
    N_WAYS = _cfg_require('GPN_V3_GMM.ipynb.cell3.main.N_WAYS')
    K_SHOT = _cfg_require('GPN_V3_GMM.ipynb.cell3.main.K_SHOT')
    Q_QUERY = _cfg_require('GPN_V3_GMM.ipynb.cell3.main.Q_QUERY')
    
    # 创建数据集
    # train_dataset = YourTrainDataset(...)
    # test_dataset = YourTestDataset(...)
    
    # 创建模型
    model = GPN_Optimized(
        attention_type=_cfg_require('GPN_V3_GMM.ipynb.cell3.main.attention_type'),
        reduction=_cfg_require('GPN_V3_GMM.ipynb.cell3.main.reduction'),
    )
    
    # 创建GMM trainer
    meta_learner = GPNTrainer(
        model,
        device=device,
        mlr=_cfg_require('GPN_V3_GMM.ipynb.cell3.main.mlr'),
        use_gmm=_cfg_require('GPN_V3_GMM.ipynb.cell3.main.use_gmm'),              # ★ GMM模式
        gmm_components=_cfg_require('GPN_V3_GMM.ipynb.cell3.main.gmm_components'),          # ★ <configured>个高斯分量
        prototypes_per_class=_cfg_require('GPN_V3_GMM.ipynb.cell3.main.prototypes_per_class'),
        cut_ratio=_cfg_require('GPN_V3_GMM.ipynb.cell3.main.cut_ratio'),
        resize=_cfg_require('GPN_V3_GMM.ipynb.cell3.main.resize'),
        save_middle=_cfg_require('GPN_V3_GMM.ipynb.cell3.main.save_middle'),
    )
    
    print(f"\nGMM配置:")
    print(f"  - 每个类的高斯分量数: {meta_learner.gmm_components}")
    print("  - 特征维度: inspect the configured model output")
    print(f"  - 总GMM参数数: {N_WAYS * meta_learner.gmm_components * feature_dim * 2}")
    
    # 训练（如果需要）
    # meta_learner.train_loop(
    #     train_dataset=train_dataset,
    #     test_dataset=test_dataset,
    # Historical configuration omitted; provide values through private configuration.
    # Historical configuration omitted; provide values through private configuration.
    # Historical configuration omitted; provide values through private configuration.
    # Historical configuration omitted; provide values through private configuration.
    # Historical configuration omitted; provide values through private configuration.
    # Historical configuration omitted; provide values through private configuration.
    # )
    
    # 或者加载预训练模型
    # Historical configuration omitted; provide values through private configuration.
    
    # 完整评估
    print("\n开始评估...")
    results = meta_learner.full_evaluation(
        test_dataset,
        n_trials=_cfg_require('GPN_V3_GMM.ipynb.cell3.main.n_trials'),
        q_query=Q_QUERY
    )
    
    # 显示结果
    print(f"\n{'='*50}")
    print("GMM模式评估结果")
    print(f"{'='*50}")
    print(f"平均准确率: {results['mean_accuracy']:.4f} ({results['mean_accuracy']*100:.2f}%)")
    print(f"标准差: {results['std_accuracy']:.4f}")
    print(f"95%置信区间: [{results['ci_lower']:.4f}, {results['ci_upper']:.4f}]")
    
    # 可视化
    # plot_all_visualizations(results, output_dir='./results/gmm')
    
    return results

if __name__ == '__main__':
    results = main()


# ============================================
# 参数调优建议
# ============================================
'\nGMM参数调优建议：\n\n<configured>. gmm_components（高斯分量数）:\n   - 建议范围: <configured>-<configured>\n   - 太少(<configured>-<configured>): 退化为单原型，表达能力不足\n   - 太多(><configured>): 过拟合风险，计算开销大\n   - 推荐起点: <configured>\n   - 根据k_shot调整: gmm_components <= k_shot\n\n<configured>. 与k_shot的关系:\n   - k_shot=<configured>: 建议 gmm_components=<configured>\n   - k_shot=<configured>: 建议 gmm_components=<configured>\n   - k_shot=<configured>: 建议 gmm_components=<configured>-<configured>\n   - 原则: 每个分量至少有<configured>-<configured>个样本支持\n\n<configured>. 计算效率考虑:\n   - GMM的计算复杂度: O(n_ways * gmm_components * feature_dim^<configured>)\n   - 对于实时应用，建议gmm_components<=<configured>\n   - 对于离线分析，可以尝试更大的值\n\n<configured>. 与其他参数的配合:\n   - cut_ratio: <configured>-<configured> (频率域裁剪)\n   - use_edge_corner: True (增强特征)\n   - mlr: <configured>-<configured> (学习率)\n\n<configured>. 训练技巧:\n   - 先用单原型或多原型模式预训练\n   - 再切换到GMM模式fine-tune\n   - 监控每个类的GMM分量分布情况\n'


# ============================================
# 调试和可视化GMM
# ============================================
"""
查看GMM参数的分布情况：
"""

def visualize_gmm_parameters(meta_learner, test_loader, n_ways=None, save_path=None):
    """
    可视化GMM参数分布
    """
    n_ways = _cfg_resolve('GPN_V3_GMM.ipynb.cell3.visualize_gmm_parameters.n_ways', n_ways)
    save_path = _cfg_resolve('GPN_V3_GMM.ipynb.cell3.visualize_gmm_parameters.save_path', save_path)
    import matplotlib.pyplot as plt
    from matplotlib.patches import Ellipse
    
    meta_learner.model.eval()
    
    # 获取一个batch
    for meta_task in test_loader:
        support_signals, support_labels, query_signals, query_labels = meta_task
        break
    
    # 提取特征
    support_signals = support_signals.to(meta_learner.device).float().permute(1, 0, 2, 3)
    support_labels = support_labels.to(meta_learner.device).flatten()
    
    with torch.no_grad():
        support_v, support_s = meta_learner.model(support_signals)
    
    # 重映射标签
    unique_labels = torch.unique(support_labels)
    label_mapping = {label.item(): i for i, label in enumerate(unique_labels)}
    remapped_labels = torch.tensor(
        [label_mapping[label.item()] for label in support_labels],
        device=meta_learner.device
    )
    
    # 估计GMM参数
    gmm_params = meta_learner.compute_gmm_parameters(
        support_v, support_s, remapped_labels, n_ways
    )
    
    means = gmm_params['means'].cpu().numpy()  # [n_ways, n_components, feature_dim]
    covariances = gmm_params['covariances'].cpu().numpy()
    weights = gmm_params['weights'].cpu().numpy()
    
    # 使用PCA降维到2D用于可视化
    from sklearn.decomposition import PCA
    
    support_v_np = support_v.cpu().numpy()
    pca = PCA(n_components=_cfg_require('GPN_V3_GMM.ipynb.cell3.visualize_gmm_parameters.n_components'))
    support_v_2d = pca.fit_transform(support_v_np)
    
    # 绘图
    fig, axes = plt.subplots(2, 4, figsize=(20, 10))
    axes = axes.flatten()
    
    for class_idx in range(n_ways):
        ax = axes[class_idx]
        
        # 绘制该类的样本
        class_mask = (remapped_labels == class_idx).cpu().numpy()
        ax.scatter(support_v_2d[class_mask, 0], support_v_2d[class_mask, 1], 
                  alpha=_cfg_require('GPN_V3_GMM.ipynb.cell3.visualize_gmm_parameters.alpha'), s=50, label='Samples')
        
        # 绘制GMM分量
        class_means_2d = pca.transform(means[class_idx])
        
        for comp_idx in range(meta_learner.gmm_components):
            mean_2d = class_means_2d[comp_idx]
            weight = weights[class_idx, comp_idx]
            
            if weight > 0.01:  # 只显示权重大于<configured>的分量
                # 绘制均值
                ax.scatter(mean_2d[0], mean_2d[1], 
                         marker='x', s=_cfg_require('GPN_V3_GMM.ipynb.cell3.visualize_gmm_parameters.size_or_budget__2'), linewidths=3,
                         color='red', label=f'Comp {comp_idx+1} (w={weight:.2f})')
                
                # 绘制协方差椭圆（简化版）
                # 注意：这里只是示意，实际的协方差椭圆需要更复杂的计算
                circle = plt.Circle(mean_2d, radius=0.5*weight, 
                                  fill=False, color='red', linestyle='--', alpha=_cfg_require('GPN_V3_GMM.ipynb.cell3.visualize_gmm_parameters.alpha__3'))
                ax.add_patch(circle)
        
        ax.set_title(f'Class {class_idx+1}')
        ax.legend()
        ax.grid(True, alpha=_cfg_require('GPN_V3_GMM.ipynb.cell3.visualize_gmm_parameters.alpha__2'))
    
    plt.suptitle(f'GMM Visualization ({meta_learner.gmm_components} components per class)')
    plt.tight_layout()
    plt.savefig(save_path, dpi=_cfg_require('GPN_V3_GMM.ipynb.cell3.visualize_gmm_parameters.size_or_budget'), bbox_inches='tight')
    plt.show()
    
    print(f"GMM可视化已保存到: {save_path}")

# 使用示例
# Historical configuration omitted; provide values through private configuration.

In [ ]:
from amgpn.config import require as _cfg_require, resolve as _cfg_resolve, format_value as _cfg_format
#主要变化：使用SiLU取代ReLU,优化模型架构，以此作为新模型迭代起点
import sys
sys.path.append("..") #相对路径或绝对路径

from pathlib import Path
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import numpy as np
from tqdm import tqdm
from amgpn.data.wavelet_episodes import *
from amgpn.legacy.v4.gmm import *
import higher
from amgpn.visualization.single import *
from thop import profile, clever_format
#约定：所有训练时使用类数<configured>抽<configured>，总样本<configured>个，每个任务中样本数支持集<configured>，查询集<configured>，
#     所有测试时使用类数<configured>抽<configured>-<configured>，总样本<configured>个，每个任务中样本数支持集<configured>,<configured>,<configured>，查询集<configured>，

# 参数设置
LOSS_FEATURE_DIM = _cfg_require('GPN_V3_GMM.ipynb.cell4.module.LOSS_FEATURE_DIM')
N_WAY = _cfg_require('GPN_V3_GMM.ipynb.cell4.module.N_WAY')
K_SHOT = _cfg_require('GPN_V3_GMM.ipynb.cell4.module.K_SHOT')
Q_QUERY = _cfg_require('GPN_V3_GMM.ipynb.cell4.module.Q_QUERY')
NUM_TASKS = _cfg_require('GPN_V3_GMM.ipynb.cell4.module.NUM_TASKS')
EPOCHS = _cfg_require('GPN_V3_GMM.ipynb.cell4.module.EPOCHS')
BLR = _cfg_require('GPN_V3_GMM.ipynb.cell4.module.BLR')
LOSSLR=_cfg_require('GPN_V3_GMM.ipynb.cell4.module.LOSSLR')
MLR = _cfg_require('GPN_V3_GMM.ipynb.cell4.module.MLR')
BATCH_SIZE =_cfg_require('GPN_V3_GMM.ipynb.cell4.module.BATCH_SIZE')
loss_alpha=_cfg_require('GPN_V3_GMM.ipynb.cell4.module.loss_alpha')
SEED = _cfg_require('GPN_V3_GMM.ipynb.cell4.module.SEED')
SAVE_MODEL_DIR = Path(_cfg_require('GPN_V3_GMM.ipynb.cell4.module.path'))
SAVE_MODEL_DIR.mkdir(parents=True, exist_ok=True)
SAVE_MODEL_PATH=SAVE_MODEL_DIR/_cfg_require('GPN_V3_GMM.ipynb.cell4.module.path__2')
LOAD_MODEL_PATH=_cfg_require('GPN_V3_GMM.ipynb.cell4.module.LOAD_MODEL_PATH')
# 数据路径
# current_path=Path(__file__).resolve().parent
dataset_path = _cfg_require('GPN_V3_GMM.ipynb.cell4.module.dataset_path')
GMM_COMPONENTS=_cfg_require('GPN_V3_GMM.ipynb.cell4.module.GMM_COMPONENTS')

    
def main():
    
    train_dataset=UAVDataset(data_dir_path=dataset_path,seed=SEED,is_train=True,train_ratio=_cfg_require('GPN_V3_GMM.ipynb.cell4.main.train_ratio'),max_sample_count=_cfg_require('GPN_V3_GMM.ipynb.cell4.main.max_sample_count'),
                            use_wavelet_denoise=_cfg_require('GPN_V3_GMM.ipynb.cell4.main.use_wavelet_denoise'), 
                             wavelet_name=_cfg_require('GPN_V3_GMM.ipynb.cell4.main.wavelet_name'), 
                             denoise_level=_cfg_require('GPN_V3_GMM.ipynb.cell4.main.denoise_level'),)
                             # Historical configuration omitted; provide values through private configuration.
    test_dataset=UAVDataset(data_dir_path=dataset_path,seed=SEED,is_train=False,train_ratio=_cfg_require('GPN_V3_GMM.ipynb.cell4.main.train_ratio__2'),max_sample_count=_cfg_require('GPN_V3_GMM.ipynb.cell4.main.max_sample_count__2'),
                            use_wavelet_denoise=_cfg_require('GPN_V3_GMM.ipynb.cell4.main.use_wavelet_denoise__2'), 
                             wavelet_name=_cfg_require('GPN_V3_GMM.ipynb.cell4.main.wavelet_name__2'), 
                             denoise_level=_cfg_require('GPN_V3_GMM.ipynb.cell4.main.denoise_level__2'),)
                             # Historical configuration omitted; provide values through private configuration.
    # Historical configuration omitted; provide values through private configuration.
    # Historical configuration omitted; provide values through private configuration.

    
    # 创建元任务划分集
# Historical configuration omitted; provide values through private configuration.
# Historical configuration omitted; provide values through private configuration.
    
    meta_train_dataset = MetaDataset(train_dataset, 8,15,5,num_tasks=NUM_TASKS)
    meta_test_dataset = MetaDataset(test_dataset, N_WAY, K_SHOT, Q_QUERY,num_tasks=_cfg_require('GPN_V3_GMM.ipynb.cell4.main.num_tasks'))
    
    # DataLoader
    train_loader = DataLoader(meta_train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True, num_workers=_cfg_require('GPN_V3_GMM.ipynb.cell4.main.num_workers'))
    test_loader = DataLoader(meta_test_dataset, batch_size=_cfg_require('GPN_V3_GMM.ipynb.cell4.main.batch_size'), shuffle=True)

    #创建模型
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # model = GPN(use_se=True)#GPN_change
    # model = GPN_Optimized(use_repvgg=True, use_convnext=True)#GPN_new
    model = GPN_Optimized(
                 attention_type=_cfg_require('GPN_V3_GMM.ipynb.cell4.main.attention_type'),
                 reduction=_cfg_require('GPN_V3_GMM.ipynb.cell4.main.reduction'),)

    params = count_parameters(model)
    
    
    x=torch.randn(2,1,_cfg_require('GPN_V3_GMM.ipynb.cell4.main.size_or_budget'),_cfg_require('GPN_V3_GMM.ipynb.cell4.main.size_or_budget__2'))
    # 前向传播
    v, s = model(x)
    print(f"Embedding features (v) shape: {v.shape}")
    print(f"Precision features (s) shape: {s.shape}")
    print(f"Precision values are positive: {(s > 0).all().item()}")

    # 带中间特征输出
    v, s, features = model(x, return_intermediate=True)
    print(f"\nIntermediate features:")
    for name, feat in features.items():
        if isinstance(feat, torch.Tensor):
            print(f"  {name}: {feat.shape}")
    input_tensor = torch.randn(1, 1, _cfg_require('GPN_V3_GMM.ipynb.cell4.main.size_or_budget__3'), _cfg_require('GPN_V3_GMM.ipynb.cell4.main.size_or_budget__4'))
    
    flops, params = profile(model, inputs=(input_tensor,))
    flops, params = clever_format([flops, params], "%.3f")
    print(f"FLOPs: {flops}, Params: {params}")
    # 创建训练器
    meta_learner = GPNTrainer(
        model,
        device=device,
        mlr=_cfg_require('GPN_V3_GMM.ipynb.cell4.main.mlr'),                    # 学习率
        use_gmm=_cfg_require('GPN_V3_GMM.ipynb.cell4.main.use_gmm'),                 # ★ 启用GMM混合高斯模式
        gmm_components=GMM_COMPONENTS,# ★ 每个类使用<configured>个高斯分量
        prototypes_per_class=_cfg_require('GPN_V3_GMM.ipynb.cell4.main.prototypes_per_class'),      # 最大原型数（向后兼容）
        crop_ratio_h=_cfg_require('GPN_V3_GMM.ipynb.cell4.main.crop_ratio_h'),crop_ratio_l=_cfg_require('GPN_V3_GMM.ipynb.cell4.main.crop_ratio_l'),                # 频率裁剪比例
        resize=_cfg_require('GPN_V3_GMM.ipynb.cell4.main.resize'),                  # 是否resize
        save_middle=_cfg_require('GPN_V3_GMM.ipynb.cell4.main.save_middle'),             # 是否保存中间最优模型
    )
    print("开始训练...")
    meta_learner.train(train_loader=train_loader,test_loader=test_loader,epochs=EPOCHS,n_ways=N_WAY,save_path=SAVE_MODEL_PATH)
    print("训练完成！")

    # meta_learner.evaluate(test_loader=test_loader,n_ways=N_WAY)
    
def inference():
    '\n    推理示例：加载已保存的模型进行预测\n    \n    !!!若报模型结构错误，检查：<configured>.引用是否为对应模型\n    '
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    # 创建模型
    test_dataset=UAVDataset(data_dir_path=_cfg_require('GPN_V3_GMM.ipynb.cell4.inference.data_dir_path'),seed=SEED,is_train=False,
                            train_ratio=_cfg_require('GPN_V3_GMM.ipynb.cell4.inference.train_ratio'),max_sample_count=_cfg_require('GPN_V3_GMM.ipynb.cell4.inference.max_sample_count'),
                            use_wavelet_denoise=_cfg_require('GPN_V3_GMM.ipynb.cell4.inference.use_wavelet_denoise'), 
                            wavelet_name=_cfg_require('GPN_V3_GMM.ipynb.cell4.inference.wavelet_name'), 
                            denoise_level=_cfg_require('GPN_V3_GMM.ipynb.cell4.inference.denoise_level'),)
                            # Historical configuration omitted; provide values through private configuration.

    # Historical configuration omitted; provide values through private configuration.
    # Historical configuration omitted; provide values through private configuration.

    # model = GPN(use_se=True)
    model = GPN_Optimized(
                 attention_type=_cfg_require('GPN_V3_GMM.ipynb.cell4.inference.attention_type'),#se,decoupled,freq_priority
                 reduction=_cfg_require('GPN_V3_GMM.ipynb.cell4.inference.reduction'),
                 )
    # 创建trainer并加载模型
    # trainer = GPNTrainer(model, device=device)
    
    meta_learner = GPNTrainer(
        model,
        device=device,
        mlr=MLR,                    # 学习率
        use_gmm=_cfg_require('GPN_V3_GMM.ipynb.cell4.inference.use_gmm'),                 # ★ 启用GMM混合高斯模式
        gmm_components=GMM_COMPONENTS,# ★ 每个类使用<configured>个高斯分量
        prototypes_per_class=_cfg_require('GPN_V3_GMM.ipynb.cell4.inference.prototypes_per_class'),      # 最大原型数（向后兼容）
        crop_ratio_h=_cfg_require('GPN_V3_GMM.ipynb.cell4.inference.crop_ratio_h'),crop_ratio_l=_cfg_require('GPN_V3_GMM.ipynb.cell4.inference.crop_ratio_l'),                # 频率裁剪比例
        resize=_cfg_require('GPN_V3_GMM.ipynb.cell4.inference.resize'),                  # 是否resize
        save_middle=_cfg_require('GPN_V3_GMM.ipynb.cell4.inference.save_middle'),             # 是否保存中间最优模型
    )
    meta_learner.load_model(LOAD_MODEL_PATH)
    
    # 评估
    results = meta_learner.full_evaluation(test_dataset, n_trials=_cfg_require('GPN_V3_GMM.ipynb.cell4.inference.n_trials'),q_query=Q_QUERY)
    plot_all_visualizations(results,output_dir=_cfg_require('GPN_V3_GMM.ipynb.cell4.inference.output_dir'))#V2_rnx+rnx_d_ft5:1_L_8_1_5
    print("推理完成")
if __name__ == '__main__':
    torch.autograd.set_detect_anomaly(True)
    # main()
    inference()
